<a href="https://colab.research.google.com/github/Bwang101/Heart-Disease-Predictor/blob/main/Heart_Disease_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import kagglehub
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Download latest version
path = kagglehub.dataset_download("amineipad/heart-disease-dataset")

print("Path to dataset files:", path)

data = pd.read_csv(path + "/heart.csv")

Using Colab cache for faster access to the 'heart-disease-dataset' dataset.
Path to dataset files: /kaggle/input/heart-disease-dataset


In [16]:
# calculate thresholds based on dataset statistics
age_thresh = data["Age"].mean()
chol_thresh = data["Cholesterol"].mean()
bp_thresh = data["BP"].mean()
thalach_thresh = data["Thallium"].mean()
oldpeak_thresh = data["ST depression"].mean()


In [17]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [18]:
data["Heart Disease"] = data["Heart Disease"].map({
    "Presence": 1,
    "Absence": 0
})

In [19]:
X = data.drop("Heart Disease", axis=1)
y = data["Heart Disease"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [21]:
class HeartModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

model = HeartModel()

In [22]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 200

for epoch in range(epochs):

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.6946289539337158
Epoch: 20 Loss: 0.6448347568511963
Epoch: 40 Loss: 0.5693878531455994
Epoch: 60 Loss: 0.4685054421424866
Epoch: 80 Loss: 0.3874105215072632
Epoch: 100 Loss: 0.3483390510082245
Epoch: 120 Loss: 0.3253451883792877
Epoch: 140 Loss: 0.3056406080722809
Epoch: 160 Loss: 0.28701624274253845
Epoch: 180 Loss: 0.2675073444843292


In [23]:
with torch.no_grad():
    predictions = model(X_test)
    predicted = (predictions > 0.5).float()
    accuracy = (predicted == y_test).float().mean()

print("Model Accuracy:", accuracy.item())

Model Accuracy: 0.9074074029922485


In [24]:
# Input format:
# [age, sex (1=male, 0=female), chest pain type (0-3),
#  resting blood pressure, cholesterol,
#  fasting blood sugar >120 (1=yes, 0=no),
#  resting ECG results (0-2),
#  max heart rate achieved,
#  exercise induced angina (1=yes, 0=no),
#  oldpeak (ST depression),
#  slope of peak exercise ST segment (0-2),
#  number of major vessels (0-3),
#  thalassemia (0-3)]

import torch

user_data = [[45, 1, 2, 130, 230, 0, 1, 150, 0, 1.0, 1, 0, 2]]

user_tensor = torch.tensor(user_data, dtype=torch.float32)

model.eval()

with torch.no_grad():
    output = model(user_tensor)
    output = torch.sigmoid(output)
    prediction = (output > 0.5).int()

In [25]:
issues = []

age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal = user_data[0]

if age > age_thresh:
    issues.append("Higher than average age")

if chol > chol_thresh:
    issues.append("Above average cholesterol")

if trestbps > bp_thresh:
    issues.append("Above average blood pressure")

if thalach < thalach_thresh:
    issues.append("Below average max heart rate")

if exang == 1:
    issues.append("Exercise-induced angina")

if oldpeak > oldpeak_thresh:
    issues.append("Above average ST depression")

if ca > 0:
    issues.append("Blocked blood vessels detected")

if thal > 1:
    issues.append("Abnormal thalassemia result")

In [26]:
if prediction.item() == 1:
    print("High risk of heart disease")
else:
    print("Low risk of heart disease")

print("\nPossible contributing factors:")

if issues:
    for issue in issues:
        print("-", issue)
else:
    print("No major risk factors detected")

High risk of heart disease

Possible contributing factors:
- Abnormal thalassemia result
